# LLM-Supported Natural Language to Bash Translation

## Load Dataset

In [1]:
# from datasets import load_dataset

# # note the config parameter, NOT the split parameter, selects the train/test data
# train_dataset = load_dataset("westenfelder/NL2SH-ALFA", "train", split="train")
# test_dataset = load_dataset("westenfelder/NL2SH-ALFA", "test", split="train")

# print(f"Train dataset size: {len(train_dataset)} rows")
# print(f"Test dataset size: {len(test_dataset)} rows")

# print("\nExample Row")
# print(f"Natural Language Task: {train_dataset[0]['nl']}")
# print(f"Bash Command: {train_dataset[0]['bash']}")
import os
from datasets import load_dataset

base = "/home/redili/github/NL2SH/nl2sh_alfa"   # 绝对路径
train_dataset = load_dataset("csv", data_files=f"{base}/train.csv", split="train")
test_dataset  = load_dataset("csv", data_files=f"{base}/test.csv",  split="train")

print(f"Train: {len(train_dataset)} rows")
print(f"Test:  {len(test_dataset)} rows")
print(f"Test cols: {test_dataset.column_names}")  # 应该是 nl/bash/bash2/difficulty
print(f"Features: {train_dataset.features}")  # 应该是 nl/bash/bash2/difficulty

/home/redili/.conda/envs/rdlpytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 40639 rows
Test:  300 rows
Test cols: ['nl', 'bash', 'bash2', 'difficulty']
Features: {'nl': Value('string'), 'bash': Value('string')}


## Load Model

In [11]:
# import torch
# import random
# from transformers import AutoTokenizer, AutoModelForCausalLM

# model_id = "meta-llama/Llama-3.2-1B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(model_id, clean_up_tokenization_spaces=False)
# tokenizer.pad_token = tokenizer.eos_token
# model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda", torch_dtype=torch.bfloat16)

# # for reproducibility
# seed = 123
# torch.manual_seed(seed)
# random.seed(seed)
# torch.cuda.manual_seed_all(seed)

# def translate(prompt, system_prompt="Your task is to translate a natural language instruction to a Bash command. You will receive an instruction in English and output a Bash command that can be run in a Linux terminal."):
#     messages = [
#         {"role": "system", "content": system_prompt},
#         {"role": "user", "content": f"{prompt}"},
#     ]

#     tokens = tokenizer.apply_chat_template(
#         messages,
#         add_generation_prompt=True,
#         tokenize=True,
#         return_tensors="pt"
#     ).to(model.device)

#     attention_mask = torch.ones_like(tokens)

#     terminators = [
#         tokenizer.eos_token_id,
#         tokenizer.convert_tokens_to_ids("<|eot_id|>")
#     ]

#     outputs = model.generate(
#         tokens,
#         attention_mask=attention_mask,
#         max_new_tokens=100,
#         eos_token_id=terminators,
#         pad_token_id=tokenizer.eos_token_id,
#         do_sample=False,
#         temperature=None,
#         top_p=None,
#     )
    
#     # remove the prompt from the output
#     response = outputs[0][tokens.shape[-1]:]
#     return tokenizer.decode(response, skip_special_tokens=True)
import torch
import random
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "/home/redili/models/qwen2.5-coder-0.5b"   # 本地路径
tokenizer = AutoTokenizer.from_pretrained(model_id, clean_up_tokenization_spaces=False)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda", dtype=torch.bfloat16)

# for reproducibility
seed = 123
torch.manual_seed(seed)
random.seed(seed)
torch.cuda.manual_seed_all(seed)

def translate(prompt, system_prompt="Your task is to translate a natural language instruction to a Bash command. You will receive an instruction in English and output a Bash command that can be run in a Linux terminal."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"{prompt}"},
    ]

    tokens_dict = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt"
    ).to(model.device)

    input_ids = tokens_dict["input_ids"]
    attention_mask = tokens_dict["attention_mask"]

    outputs = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=100,
        do_sample=False,
        temperature=None,
        top_p=None,
        top_k=None,
    )

    response = outputs[0][input_ids.shape[-1]:]
    return tokenizer.decode(response, skip_special_tokens=True)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1660.85it/s]


In [12]:
import re

# strip markdown formatting
def parse_bash(text):
    patterns = [
        r"```bash\s*(.*?)\s*```",
        r"```(.*?)```",
        r"`(.*?)`",
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            return match.group(1).strip()
    
    return text

In [14]:
# example usage
natural_language_task = train_dataset[0]["nl"]
ground_truth_command = train_dataset[0]["bash"]
model_output = translate(natural_language_task)
model_command = parse_bash(model_output)

print(f"Natural Language Task: {natural_language_task}")
print(f"Ground Truth Command: {ground_truth_command}")
print(f"Model Command: {model_command}")

Natural Language Task: show the free space on all filesystems
Ground Truth Command: df -h
Model Command: df


## Benchmark Model

In [16]:
from icalfa import submit_command
from tqdm import tqdm

num_correct = 0
total = len(test_dataset)

for index, row in tqdm(enumerate(test_dataset), total=total):
    natural_language_task = row['nl']
    model_output = translate(natural_language_task)
    model_command = parse_bash(model_output)
    num_correct += submit_command(index=index, command=model_command, eval_mode="embed", eval_param=0.75)

print(f"Model Accuracy: {(num_correct/total):0.2f}")

  0%|          | 1/300 [00:00<03:33,  1.40it/s]

Benchmark Error: Error while fetching server API version: ('Connection aborted.', PermissionError(13, 'Permission denied'))
Attempting to reset container.
Failed to reset container. Recommend manually rebuilding Docker containers.
Error results in score of zero.


  1%|          | 2/300 [00:01<03:56,  1.26it/s]

Benchmark Error: Error while fetching server API version: ('Connection aborted.', PermissionError(13, 'Permission denied'))
Attempting to reset container.
Failed to reset container. Recommend manually rebuilding Docker containers.
Error results in score of zero.


  1%|          | 3/300 [00:02<04:17,  1.15it/s]

...

Benchmark Error: Error while fetching server API version: ('Connection aborted.', PermissionError(13, 'Permission denied'))
Attempting to reset container.
Failed to reset container. Recommend manually rebuilding Docker containers.
Error results in score of zero.


 63%|██████▎   | 189/300 [03:20<02:12,  1.19s/it]

Benchmark Error: Error while fetching server API version: ('Connection aborted.', PermissionError(13, 'Permission denied'))
Attempting to reset container.
Failed to reset container. Recommend manually rebuilding Docker containers.
Error results in score of zero.


 63%|██████▎   | 189/300 [03:21<01:58,  1.06s/it]


KeyboardInterrupt: 